# 09 — 변수 선택 재설계 (도메인 그룹핑 + 예측 최적화 + 성능 분포 정당화)

> 기말 플랜 `docs/final_presentation_plan.md` **§2.6** 구현 · 교수님 피드백 **F1·F3·F4** 응답
> **프로젝트 우선순위: 예측 1순위** — 변수 선택을 SHAP(해석)이 아니라 **성능 분포**로 정당화

## 목표
1. **(F3)** 22개 후보를 **5개 도메인 그룹**으로 묶고 대표 변수 선정
2. **(F1)** **VIF 사전 제거 vs 미제거** 변수집합 A/B/C ablation — 예측 성능 비교
3. **(⑥)** 성능 분포 정당화: (a) 랜덤 서브셋 벤치마크 히스토그램, (b) 변수별 한계기여
4. 예측/해석 후보셋 확정 → `data/processed/features_v3_candidate.csv`

## ⚠️ 정직성 가드 (plan §2.6 ⑥)
- 히스토그램은 **선택의 사후 검증용**이지 *선택 도구 아님* (최고 서브셋 cherry-pick = data snooping)
- 분포 sweep은 **경량 프록시**(단일 XGBoost 점예측)로, 최종 선택셋만 full 파이프라인(노트북 11)에서 재검증

## 의존
- 입력: `data/interim/wide_daily_filled.csv`(22후보), `data/processed/features_v2_no_leak.csv`(파생5)
- 후속: `11_v3_results.ipynb`(확정셋으로 XGB/LSTM 비교), `12_pred_interval_v3.ipynb`

---

## 0. 환경 설정

In [ ]:
# === 의존성 체크 + import ===
import importlib.util, subprocess, sys
REQUIRED = {'xgboost':'xgboost','sklearn':'scikit-learn','matplotlib':'matplotlib','seaborn':'seaborn'}
for _imp, _pip in REQUIRED.items():
    if importlib.util.find_spec(_imp) is None:
        print(f'  Installing {_pip} ...'); subprocess.check_call([sys.executable,'-m','pip','install','-q',_pip])

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
FIG_DIR  = PROJECT_ROOT / 'reports' / 'figures' / 'v3'
DOCS_DIR = PROJECT_ROOT / 'docs'
FIG_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'kr_treasury_10y'
SEED = 42; np.random.seed(SEED)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 180)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('FIG_DIR     :', FIG_DIR.relative_to(PROJECT_ROOT))

---

## 1. 데이터 로드 — 후보 풀 + 파생변수

- `wide_daily_filled.csv`: 22개 거시 후보(국내 금리 기간구조·kospi 등 포함)
- `features_v2_no_leak.csv`: 중간 v2 파생 5개 포함

In [ ]:
wide = pd.read_csv(DATA_DIR/'interim'/'wide_daily_filled.csv', index_col='date', parse_dates=['date'])
feat_v2 = pd.read_csv(DATA_DIR/'processed'/'features_v2_no_leak.csv', index_col='date', parse_dates=['date'])

DERIVED = ['spread_10y_t1','delta_us10y_t1','delta_vix_t1','delta_dxy_t1','crisis_dummy']
derived_avail = [c for c in DERIVED if c in feat_v2.columns]

# master: 후보 풀(wide) + 파생(feat_v2) 결합
master = wide.join(feat_v2[derived_avail], how='left')
target_delta = wide[TARGET].diff().rename('target_dy')

print('wide 후보 컬럼 (%d):' % wide.shape[1]); print(' ', list(wide.columns))
print('\n파생 확보 (%d/%d):' % (len(derived_avail), len(DERIVED)), derived_avail)
print('master.shape:', master.shape, '| 기간:', master.index.min().date(), '~', master.index.max().date())

---

## 2. 도메인 5개 그룹 정의 (F3 · plan §2.6①)

> VIF로 '제거'가 아니라, **경제 채널로 묶은 뒤 대표를 선택**. (교수님 피드백 핵심)

In [ ]:
# plan §2.6① — 그룹 배정 (대표는 **굵게** 표기한 1차안)
GROUPS = {
    'G1 글로벌금리':     ['us_treasury_10y','delta_us10y_t1','us_fed_funds','us_breakeven_10y'],
    'G2 한미연계/EM':    ['spread_10y_t1','dxy','delta_dxy_t1'],
    'G3 국내기간구조/정책':['kr_treasury_3y','kr_base_rate'],   # 복원후보: kr_treasury_5y, kr_treasury_1y
    'G4 위험심리':       ['vix','delta_vix_t1','sp500'],
    'G5 위기레짐':       ['crisis_dummy'],
}
REP = {'G1 글로벌금리':'delta_us10y_t1','G2 한미연계/EM':'spread_10y_t1',
       'G3 국내기간구조/정책':'kr_treasury_3y','G4 위험심리':'delta_vix_t1','G5 위기레짐':'crisis_dummy'}

rows = []
for g, members in GROUPS.items():
    avail = [m for m in members if m in master.columns]
    miss  = [m for m in members if m not in master.columns]
    rows.append({'그룹':g, '대표':REP[g], '배정변수(확보)':', '.join(avail), '누락':', '.join(miss) or '-'})
groups_df = pd.DataFrame(rows)
print(groups_df.to_string(index=False))

---

## 3. 변수집합 A/B/C 정의 (F1 ablation · plan §2.6②)

- **A**: 중간 v2 선택셋 (VIF<10 + Granger 통과분 + 파생5)
- **B**: **VIF 미적용 확장셋** — A + 공선 변수(kr 5Y/1Y) + kospi(Granger 탈락) 복원
- **C**: **그룹 대표 + 규제** (최소·해석 친화)

> 가설: **(B)/(C) ≥ (A)** 면 VIF 사전 제거가 예측에 손해였음을 입증

In [ ]:
# A: 중간 v2 — 8 base(kospi 제외) + 파생 5
BASE_V2 = ['kr_treasury_3y','us_treasury_10y','us_breakeven_10y','dxy',
           'kr_base_rate','us_fed_funds','vix','sp500']
SET_A = [c for c in BASE_V2 + derived_avail if c in master.columns]

# B: VIF 미적용 확장 — A + 복원후보(공선 KR tenor + kospi)
RESTORE = ['kr_treasury_5y','kr_treasury_1y','kospi']
SET_B = SET_A + [c for c in RESTORE if c in master.columns]

# C: 그룹 대표만
SET_C = [REP[g] for g in GROUPS if REP[g] in master.columns]

for name, s in [('A (중간 v2)', SET_A), ('B (VIF미적용 확장)', SET_B), ('C (그룹대표)', SET_C)]:
    print(f'SET_{name}: {len(s)}개')
    print('  ', s)

---

## 4. 경량 walk-forward 평가 함수 (성능 프록시)

> ⚠️ **프록시**: 단일 XGBoost 점예측 + t-1 lag로 방향정확도만 측정 (빠른 분포 sweep용).
> lag/rolling 162피처·CQR·분위수 full 파이프라인은 **노트북 11**에서 검증.
> TODO: 08_v2_no_leak_pipeline의 walk-forward fold 경계와 정합 확인.

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit

def eval_dir_acc(cols, n_splits=3, seed=SEED):
    """feat 컬럼들의 t-1 값으로 익일 target_dy 부호를 예측 → walk-forward 평균 방향정확도(프록시)."""
    cols = [c for c in cols if c in master.columns]
    if not cols:
        return np.nan
    X = master[cols].shift(1)                      # 누수 차단: 모든 예측변수 t-1
    df = pd.concat([X, target_delta], axis=1).dropna()
    Xv, yv = df[cols].values, df['target_dy'].values
    tscv = TimeSeriesSplit(n_splits=n_splits)
    accs = []
    for tr, te in tscv.split(Xv):
        m = XGBRegressor(n_estimators=120, max_depth=3, learning_rate=0.05,
                         subsample=0.8, reg_lambda=1.0, random_state=seed, n_jobs=-1)
        m.fit(Xv[tr], yv[tr])
        pred = m.predict(Xv[te])
        accs.append(float(np.mean(np.sign(pred) == np.sign(yv[te]))))
    return float(np.mean(accs))

# 스모크 테스트
print('SET_C dir acc (proxy):', round(eval_dir_acc(SET_C), 4))

---

## 5. A/B/C ablation 비교 (F1)

In [ ]:
ablation = pd.DataFrame({
    '집합': ['A 중간v2','B VIF미적용확장','C 그룹대표'],
    'n_features': [len(SET_A), len(SET_B), len(SET_C)],
    'dir_acc(proxy)': [eval_dir_acc(SET_A), eval_dir_acc(SET_B), eval_dir_acc(SET_C)],
})
print(ablation.to_string(index=False))

fig, ax = plt.subplots(figsize=(7,4))
ax.bar(ablation['집합'], ablation['dir_acc(proxy)'], color=['steelblue','darkorange','seagreen'], alpha=0.8)
ax.axhline(0.5, color='red', ls='--', lw=1, label='무작위 0.5')
ax.set_ylabel('walk-forward dir acc (proxy)'); ax.set_ylim(0.45, 0.65)
ax.set_title('F1 ablation — VIF 사전제거(A) vs 미제거(B) vs 그룹대표(C)'); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR/'09_ablation_ABC.png', dpi=120, bbox_inches='tight'); plt.show()
# TODO: 해석 — (B)/(C)가 (A) 이상이면 'VIF 무조건 제거 불필요' 근거. 최종 판단은 11에서 full 파이프라인 재확인.

---

## 6. ⑥(a) 랜덤 서브셋 벤치마크 분포

> 후보 풀에서 동일 크기 부분집합을 무작위로 뽑아 성능 분포를 그리고, **우리 도메인 선택셋이 우측 꼬리(상위 백분위)**에 있음을 확인.
> ⚠️ **검증용** — 최고 서브셋을 선택에 쓰지 않는다(data snooping 방지).

In [ ]:
# 후보 풀: master 수치형 컬럼 중 타겟 제외
CAND_POOL = [c for c in master.columns if c != TARGET and master[c].notna().mean() > 0.9]
k = len(SET_C)                 # 부분집합 크기 = 그룹대표 수
N = 150                        # 경량: 150회 (필요시 ↑)

rng = np.random.default_rng(SEED)
sub_scores = []
for _ in range(N):
    cols = list(rng.choice(CAND_POOL, size=k, replace=False))
    s = eval_dir_acc(cols)
    if not np.isnan(s): sub_scores.append(s)
sub_scores = np.array(sub_scores)

our = eval_dir_acc(SET_C)
pct = float((sub_scores < our).mean()*100)

fig, ax = plt.subplots(figsize=(8,4))
ax.hist(sub_scores, bins=25, color='lightgray', edgecolor='gray')
ax.axvline(our, color='seagreen', lw=2, label=f'그룹대표 C = {our:.3f} (상위 {100-pct:.0f}%)')
ax.axvline(np.median(sub_scores), color='black', ls=':', lw=1, label=f'랜덤 중앙값 {np.median(sub_scores):.3f}')
ax.set_xlabel('walk-forward dir acc (proxy)'); ax.set_ylabel('빈도')
ax.set_title(f'랜덤 {k}-변수 서브셋 {len(sub_scores)}개 성능 분포 vs 도메인 선택'); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR/'09_random_subset_hist.png', dpi=120, bbox_inches='tight'); plt.show()
print(f'도메인 선택 C는 랜덤 분포의 {pct:.1f} 백분위')

---

## 7. ⑥(b) 변수별 한계 기여 (LOO / add-one-in)

- **LOO**: 선택셋에서 1개 제거 시 성능 하락폭 = 그 변수의 한계 가치
- **add-one-in**: 제외 변수(kospi 등) 추가 시 성능 변화 = 복원 가치 (없으면 제외 정당화)

In [ ]:
base_set = SET_A
base_acc = eval_dir_acc(base_set)

loo = []
for c in base_set:
    acc = eval_dir_acc([x for x in base_set if x != c])
    loo.append({'변수':c, '구분':'LOO(제거)', 'Δdir_acc': base_acc - acc})   # +면 그 변수가 기여

excluded = [c for c in CAND_POOL if c not in base_set]
addin = []
for c in excluded:
    acc = eval_dir_acc(base_set + [c])
    addin.append({'변수':c, '구분':'add-in(추가)', 'Δdir_acc': acc - base_acc})  # +면 추가 가치

contrib = pd.DataFrame(loo + addin).sort_values('Δdir_acc', ascending=False)
print('base dir acc:', round(base_acc,4))
print(contrib.to_string(index=False))

fig, ax = plt.subplots(figsize=(9,6))
colors = ['seagreen' if v>=0 else 'indianred' for v in contrib['Δdir_acc']]
ax.barh(contrib['변수']+' ('+contrib['구분'].str[:3]+')', contrib['Δdir_acc'], color=colors, alpha=0.8)
ax.axvline(0, color='black', lw=0.8); ax.set_xlabel('Δ dir acc (proxy)')
ax.set_title('변수별 한계 기여 — LOO + add-one-in'); ax.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.savefig(FIG_DIR/'09_marginal_contrib.png', dpi=120, bbox_inches='tight'); plt.show()
# TODO: kospi add-in이 ~0 이하이면 'Granger 탈락을 성능으로 재확인' 서사 확보.

---

## 8. 결론 — 예측/해석 후보셋 확정 + 정직성 가드

> 도메인 근거 + ablation + 분포를 **종합**해 결정. 분포에서 최고점을 cherry-pick 하지 않는다.

In [ ]:
# TODO: 위 결과를 보고 최종 확정 (아래는 1차안 placeholder)
PRED_SET   = SET_B   # 예측용: VIF 미제거 확장 (성능 우선) — 11에서 full 재검증
INTERP_SET = SET_C   # 해석용: 그룹 대표 (저공선성·SHAP 안정)

print('확정(1차안)'); print('  PRED_SET  (%d):'%len(PRED_SET), PRED_SET)
print('  INTERP_SET(%d):'%len(INTERP_SET), INTERP_SET)

out = master[[c for c in PRED_SET if c in master.columns]].copy()
out[TARGET] = wide[TARGET]
out_path = DATA_DIR/'processed'/'features_v3_candidate.csv'
out.to_csv(out_path)
print('\n💾 저장:', out_path.relative_to(PROJECT_ROOT), out.shape)

---

## 9. 다음 단계

- [ ] §5 ablation / §6 분포 / §7 한계기여 결과 검토 후 **§8 PRED_SET·INTERP_SET 최종 확정**
- [ ] `11_v3_results.ipynb` — 확정 `features_v3_candidate.csv`로 **XGBoost·LSTM full 파이프라인** 재실행 + DM test
- [ ] `12_pred_interval_v3.ipynb` — v3 기반 예측구간(§3.1)
- [ ] (선택·Tier3) `10_shap_dispersion.ipynb` — 군집 SHAP·interaction

### 정직성 가드 재확인
- 분포·LOO는 **검증용**, 선택은 도메인 1차 + 성능 보조
- 프록시(점예측 dir acc)와 full(분위수+CQR)의 차이는 11에서 정합 확인
